In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp
import jax

In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, lax

def generate_simulation(
    fcn_init,
    d=2,
    N=1000,
    J=100,
    dt=1e-2,
    D=1.0,
    a=2.0,
    r0=1.0,
    avg=True,           # apply 1/N averaging
    soften=1e-2,        # softening radius (fraction of r0)
    cutoff=True,        # use finite interaction cutoff for speed
    rcut_factor=5.0,    # r_cut = r0 + rcut_factor/a  (typical: 4–6)
    dtype=jnp.float64,  # float32 is much faster on GPU/TPU and fine for most sims
):
    """
    First-order interacting system with Morse interaction (gradient-flow style):
      x_i' = (1/N) * sum_{j != i} (dG/dr) * (x_j - x_i) / ||x_j - x_i||,
      G(r) = D ( e^{-2a(r-r0)} - 2 e^{-a(r-r0)} ).
    """

    invN = (1.0 / N) if avg else 1.0
    r_min = jnp.asarray(soften * max(r0, 1e-12), dtype=dtype)
    r2_min = r_min**2
    rcut = r0 + rcut_factor / a if cutoff else jnp.inf

    # Precompute diagonal mask once (inside the closure but outside jit call-sites)
    eye = jnp.eye(N, dtype=dtype)
    offdiag_mask = (1.0 - eye).astype(dtype)

    @jit
    def morse_force(x):
        # x: (N, d)
        # r_ij = x_j - x_i
        r_ij = x[jnp.newaxis, :, :] - x[:, jnp.newaxis, :]      # (N, N, d)
        r2 = jnp.sum(r_ij * r_ij, axis=-1)                      # (N, N)
        r2 = jnp.maximum(r2, r2_min)
        r = jnp.sqrt(r2)

        # dG/dr = 2 a D ( e^{-a(r-r0)} - e^{-2a(r-r0)} )
        exp1 = jnp.exp(-a * (r - r0))
        exp2 = exp1 * exp1
        dGdr = 2.0 * a * D * (exp1 - exp2)

        # g = (dG/dr) / r, zero self-terms
        g = (dGdr / r) * offdiag_mask

        # Optional cutoff: zero out pairs with r >= r_cut
        if cutoff:
            g = jnp.where(r < rcut, g, 0.0)

        # sum_j g_ij * r_ij
        f = jnp.sum(g[:, :, None] * r_ij, axis=1) * invN        # (N, d)
        return f.astype(dtype)

    # Single Euler step, JIT friendly
    @jit
    def euler_step(x, _):
        x_next = x + dt * morse_force(x)
        return x_next, x_next

    def simulations():
        # initial condition to jnp
        x0 = fcn_init(N, d)
        x0 = jnp.asarray(x0, dtype=dtype)
        if x0.shape != (N, d):
            raise ValueError(f"Incorrect initial condition shape: {x0.shape} != {(N, d)}")

        # Run scanned Euler steps: returns all states excluding x0; we’ll prepend x0
        xT, traj = lax.scan(euler_step, x0, None, length=J-1)
        X_save = jnp.concatenate([x0[None, ...], traj], axis=0)  # (J, N, d)
        return X_save

    # Return JIT’ed force and a compiled simulation function
    return morse_force, jit(simulations)


In [ ]:
def fcn_init(N,d):
    seed = 42
    rng = jax.random.PRNGKey(seed)
    N1 = N // 2
    N2 = N-N1
    x0 = jax.random.normal(rng, shape=(N1, d)) + 1
    x1 = jax.random.normal(rng, shape=(N2, d)) - 1
    return jnp.concatenate([x0, x1], axis=0)

In [ ]:
fcn_lj_force,simulations = generate_simulation(fcn_init,N=10000,d=2)

In [ ]:
X_save  = simulations()

In [ ]:
X_save.shape

In [ ]:
time= 0
plt.hist2d(
    X_save[time,:,0],X_save[time,:,1], density=True,bins=50
)

In [ ]:
time= 100
plt.hist2d(
    X_save[time,:,0],X_save[time,:,1], density=True,bins=100
);

In [ ]:
plt.plot(np.var(X_save,axis=1))